# Preprocessing — Telco Customer Churn

Turns the raw CSV into a clean, fully numeric train/test split ready for modeling. Steps follow directly from the findings in `01_eda.ipynb`.

## 1. Setup & Load Raw Data

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

pd.set_option("display.max_columns", None)

DATA_DIR = Path.cwd().resolve().parent / "data"
raw_path = DATA_DIR / "raw" / "Telco-Customer-Churn.csv"

df = pd.read_csv(raw_path)
print(df.shape)
df.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Drop Identifier Column

`customerID` is a unique key with no predictive value.

In [2]:
df = df.drop(columns=["customerID"])
df.shape

(7043, 20)

## 3. Fix `TotalCharges`

Loaded as text because 11 brand-new customers (`tenure == 0`) have blank strings instead of `0`. Coerce to numeric and fill those with `0`.

In [3]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
n_missing = df["TotalCharges"].isna().sum()
print(f"TotalCharges nulls after coercion: {n_missing}")

df["TotalCharges"] = df["TotalCharges"].fillna(0)
assert df["TotalCharges"].isna().sum() == 0
df["TotalCharges"].describe()

TotalCharges nulls after coercion: 11


count    7043.000000
mean     2279.734304
std      2266.794470
min         0.000000
25%       398.550000
50%      1394.550000
75%      3786.600000
max      8684.800000
Name: TotalCharges, dtype: float64

## 4. Collapse Redundant Categories

`"No internet service"` (6 columns) and `"No phone service"` (1 column) are fully implied by `InternetService == "No"` / `PhoneService == "No"`. Collapsing them to `"No"` turns these into true binary columns without losing information.

In [4]:
internet_dependent_cols = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies",
]
for col in internet_dependent_cols:
    df[col] = df[col].replace("No internet service", "No")

df["MultipleLines"] = df["MultipleLines"].replace("No phone service", "No")

for col in internet_dependent_cols + ["MultipleLines"]:
    print(col, "->", df[col].unique())

OnlineSecurity -> <StringArray>
['No', 'Yes']
Length: 2, dtype: str
OnlineBackup -> <StringArray>
['Yes', 'No']
Length: 2, dtype: str
DeviceProtection -> <StringArray>
['No', 'Yes']
Length: 2, dtype: str
TechSupport -> <StringArray>
['No', 'Yes']
Length: 2, dtype: str
StreamingTV -> <StringArray>
['No', 'Yes']
Length: 2, dtype: str
StreamingMovies -> <StringArray>
['No', 'Yes']
Length: 2, dtype: str
MultipleLines -> <StringArray>
['No', 'Yes']
Length: 2, dtype: str


## 5. Encode Binary Columns (Yes/No → 1/0)

`SeniorCitizen` is already `0/1`; the rest are text `Yes/No` (or `Male/Female` for `gender`).

In [5]:
binary_cols = [
    "gender", "Partner", "Dependents", "PhoneService", "PaperlessBilling",
    "MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "Churn",
]

binary_maps = {
    "gender": {"Female": 0, "Male": 1},
}
default_map = {"No": 0, "Yes": 1}

for col in binary_cols:
    mapping = binary_maps.get(col, default_map)
    df[col] = df[col].map(mapping)

df[binary_cols].head()

,gender,Partner,Dependents,PhoneService,PaperlessBilling,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Churn
0,0,1,0,0,1,0,0,1,0,0,0,0,0
1,1,0,0,1,0,0,1,0,1,0,0,0,0
2,1,0,0,1,1,0,1,1,0,0,0,0,1
3,1,0,0,0,0,0,1,0,1,1,0,0,0
4,0,0,0,1,1,0,0,0,0,0,0,0,1


In [6]:
assert df[binary_cols].isna().sum().sum() == 0, "Unexpected category found during binary encoding"
print("All binary columns encoded cleanly, no NaNs introduced.")

All binary columns encoded cleanly, no NaNs introduced.


## 6. Ordinal-Encode `Contract`

EDA showed churn rate strictly decreases from month-to-month → one year → two year, so this is a genuinely ordinal feature rather than a nominal one.

In [7]:
contract_order = [["Month-to-month", "One year", "Two year"]]
ordinal_encoder = OrdinalEncoder(categories=contract_order)
df["Contract"] = ordinal_encoder.fit_transform(df[["Contract"]])
df["Contract"].value_counts().sort_index()

Contract
0.0    3875
1.0    1473
2.0    1695
Name: count, dtype: int64

## 7. One-Hot Encode Remaining Nominal Columns

`InternetService` (3 categories) and `PaymentMethod` (4 categories) have no natural order.

In [8]:
nominal_cols = ["InternetService", "PaymentMethod"]
df = pd.get_dummies(df, columns=nominal_cols, prefix=nominal_cols, dtype=int)
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,InternetService_DSL,InternetService_Fiber optic,InternetService_No,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,0,0,1,0,0,0,0,0.0,1,29.85,29.85,0,1,0,0,0,0,1,0
1,1,0,0,0,34,1,0,1,0,1,0,0,0,1.0,0,56.95,1889.50,0,1,0,0,0,0,0,1
2,1,0,0,0,2,1,0,1,1,0,0,0,0,0.0,1,53.85,108.15,1,1,0,0,0,0,0,1
3,1,0,0,0,45,0,0,1,0,1,1,0,0,1.0,0,42.30,1840.75,0,1,0,0,1,0,0,0
4,0,0,0,0,2,1,0,0,0,0,0,0,0,0.0,1,70.70,151.65,1,0,1,0,0,0,1,0


In [9]:
print(f"Final shape: {df.shape}")
print(f"All numeric now: {(df.dtypes != object).all() and (df.dtypes.astype(str) != 'str').all()}")
df.dtypes

Final shape: (7043, 25)
All numeric now: True


gender                                       int64
SeniorCitizen                                int64
Partner                                      int64
Dependents                                   int64
tenure                                       int64
PhoneService                                 int64
MultipleLines                                int64
OnlineSecurity                               int64
OnlineBackup                                 int64
DeviceProtection                             int64
TechSupport                                  int64
StreamingTV                                  int64
StreamingMovies                              int64
Contract                                   float64
PaperlessBilling                             int64
MonthlyCharges                             float64
TotalCharges                               float64
Churn                                        int64
InternetService_DSL                          int64
InternetService_Fiber optic    

## 8. Train / Test Split

Split **before** fitting the scaler, and stratify on `Churn` since it's imbalanced (~26.5% positive).

In [10]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print("Train churn rate:", y_train.mean().round(4))
print("Test churn rate: ", y_test.mean().round(4))

Train: (5634, 24), Test: (1409, 24)
Train churn rate: 0.2654
Test churn rate:  0.2654


## 9. Scale Numeric Features

Fit `StandardScaler` on the **training set only**, then apply the same transform to test — prevents test-set information leaking into the scaling statistics.

In [11]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

X_train[numeric_cols].describe()

,tenure,MonthlyCharges,TotalCharges
count,5.634000e+03,5.634000e+03,5.634000e+03
mean,-1.008935e-17,-2.402527e-16,2.522338e-17
std,1.000089e+00,1.000089e+00,1.000089e+00
min,-1.322329e+00,-1.544028e+00,-1.008922e+00
25%,-9.559779e-01,-9.711977e-01,-8.321009e-01
50%,-1.418632e-01,1.848336e-01,-3.968446e-01
75%,9.164859e-01,8.319124e-01,6.741944e-01
max,1.608483e+00,1.785939e+00,2.801869e+00


## 10. Save Processed Data & Scaler

In [12]:
processed_dir = DATA_DIR / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

train_out = X_train.copy()
train_out["Churn"] = y_train.values
test_out = X_test.copy()
test_out["Churn"] = y_test.values

train_out.to_csv(processed_dir / "train.csv", index=False)
test_out.to_csv(processed_dir / "test.csv", index=False)

import joblib

models_dir = Path.cwd().resolve().parent / "models"
models_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(scaler, models_dir / "scaler.joblib")

print("Saved:")
print(" -", processed_dir / "train.csv", train_out.shape)
print(" -", processed_dir / "test.csv", test_out.shape)
print(" -", models_dir / "scaler.joblib")

Saved:


 - C:\ProjectsOrgnisation-churn-prediction-and-Q-A\data\processed\train.csv (5634, 25)
 - C:\ProjectsOrgnisation-churn-prediction-and-Q-A\data\processed\test.csv (1409, 25)
 - C:\ProjectsOrgnisation-churn-prediction-and-Q-A\models\scaler.joblib


## Summary

- Dropped `customerID`; fixed 11 blank `TotalCharges` values (tenure = 0 → filled with 0).
- Collapsed redundant `"No internet/phone service"` labels into `"No"` across 7 columns.
- Encoded 13 binary columns to `0/1`, `Contract` ordinally (order matches its churn-rate trend), and one-hot encoded `InternetService` / `PaymentMethod`.
- Split 80/20 stratified on `Churn` **before** scaling to avoid leakage; scaled `tenure`, `MonthlyCharges`, `TotalCharges` with a `StandardScaler` fit only on the training set.
- Saved `data/processed/train.csv`, `data/processed/test.csv`, and `models/scaler.joblib`.

**Next:** `03_model_training.ipynb`.